# SymFormer Training Notebook

Universal Symmetry-Aware Medical Segmentation

Train + Evaluate + Visualize: all in one.

In [ ]:
# Clone repo (Colab) and install dependencies
!git clone https://github.com/hoangtung386/Universal_Symmetry-Aware_Medical_Segmentation.git
%cd Universal_Symmetry-Aware_Medical_Segmentation
!pip install -e .

In [ ]:
import os, sys, json
import torch
import numpy as np
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0, 'font.size': 11})

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

## 1. Imports — All Modules

In [ ]:
from configs.config import TrainingConfig, load_config
from datasets.base import BaseDataset
from datasets.cpaisd import CPAISDDataset
from datasets.cpaisd_enhanced import CPAISDEnhancedDataset
from datasets.brats import BraTSDataset
from datasets.factory import get_dataset_class
from models.symformer import SymFormer
from models.conditioned_symformer import ConditionedSymFormer
from models.components import AlignmentNetwork, EncoderBlock3D, AdaptiveFusion
from models.losses import StrokeLoss, TverskyLoss, SymFormerLoss, create_ablation_losses
from models.bottleneck import get_bottleneck, BaseBottleneck, SymmetryAwareBottleneck, MambaBottleneckWrapper
from models.decoder.hvt import HVTDecoder, DecoderBlock, kMaXBlock
from models.decoder.kan import KANDecoderHead, KANHVTDecoder, EfficientKANLayer, RationalKANLayer
from models.layers.conditioning import ClinicalConditionEncoder, ConditionalCrossAttention
from training.trainer import SymFormerTrainer
from evaluation.evaluator import Evaluator
from evaluation.metrics import MetricCalculator
from evaluation.complexity import get_model_complexity
from utils.data_utils import compute_class_weights
from utils.transforms import apply_aligned_mask
from torch.utils.data import DataLoader

print("All imports successful \u2705")

## 2. Data Exploration

In [ ]:
DATASET = "cpaisd"
SPLIT = "train"

config = load_config(DATASET)
dataset_class = get_dataset_class(config.DATASET_NAME)
root_path = config.DATA_PATHS.get(config.DATASET_NAME, config.BASE_PATH)

dataset = dataset_class(dataset_root=root_path, split=SPLIT, T=config.T, config=config)
CLASS_NAMES = {0: 'Background', 1: 'Core', 2: 'Penumbra', 3: 'Necrosis'}
print(f"Dataset: {DATASET} ({SPLIT}), Samples: {len(dataset)}, Size: {config.IMAGE_SIZE}")

In [ ]:
pixel_counts = Counter()
sample_has_class = Counter()
for idx in range(min(len(dataset), 200)):
    _, mask, _ = dataset[idx]
    msk = mask.numpy() if torch.is_tensor(mask) else mask
    for c in range(config.NUM_CLASSES):
        cnt = int((msk == c).sum())
        pixel_counts[c] += cnt
        if cnt > 0:
            sample_has_class[c] += 1

total_px = sum(pixel_counts.values())
print(f"{'Class':<20} {'Pixels':>12} {'%':>8} {'Samples':>8}")
print("-" * 50)
for c in range(config.NUM_CLASSES):
    pct = pixel_counts[c] / total_px * 100
    print(f"{CLASS_NAMES.get(c, f'C{c}'):<20} {pixel_counts[c]:>12,d} {pct:>7.2f}% {sample_has_class[c]:>8}")

In [ ]:
colors = ['#555555', '#e74c3c', '#2ecc71', '#3498db'][:config.NUM_CLASSES]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
labels = [CLASS_NAMES.get(c, f'C{c}') for c in range(config.NUM_CLASSES)]
for ax, data, title in [(ax1, [pixel_counts[c] for c in range(config.NUM_CLASSES)], 'Pixel Count'),
                         (ax2, [sample_has_class[c] for c in range(config.NUM_CLASSES)], 'Samples with Class')]:
    bars = ax.bar(labels, data, color=colors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    for b, val in zip(bars, data):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(data)*0.01,
                f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
cmap = ListedColormap(colors)
idxs = np.random.choice(len(dataset), min(4, len(dataset)), replace=False)
fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 3*len(idxs)))
if len(idxs) == 1: axes = axes.reshape(1, -1)
leg = [mpatches.Patch(color=cmap(i), label=CLASS_NAMES.get(i, f'C{i}')) for i in range(1, config.NUM_CLASSES)]
for row, idx in enumerate(idxs):
    img, msk, _ = dataset[int(idx)]
    img_np = img.numpy() if torch.is_tensor(img) else img
    msk_np = msk.numpy() if torch.is_tensor(msk) else msk
    if img_np.ndim == 3: img_disp = img_np[img_np.shape[0]//2]
    else: img_disp = img_np
    axes[row, 0].imshow(img_disp, cmap='gray'); axes[row, 0].axis('off')
    axes[row, 1].imshow(msk_np, cmap=cmap, vmin=0, vmax=config.NUM_CLASSES-1); axes[row, 1].axis('off')
    axes[row, 2].imshow(img_disp, cmap='gray'); axes[row, 2].imshow(msk_np, cmap=cmap, alpha=0.4, vmin=0, vmax=config.NUM_CLASSES-1); axes[row, 2].axis('off')
axes[0,0].set_title('CT'); axes[0,1].set_title('Mask'); axes[0,2].set_title('Overlay')
fig.legend(handles=leg, loc='lower center', ncol=min(4, config.NUM_CLASSES-1), fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.97]); plt.show()

In [ ]:
pixel_vals = []
for idx in range(min(50, len(dataset))):
    img, _, _ = dataset[idx]
    img_np = img.numpy() if torch.is_tensor(img) else img
    if img_np.ndim == 3: img_np = img_np[img_np.shape[0]//2]
    pixel_vals.extend(img_np.flatten().tolist())
pv = np.array(pixel_vals)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.hist(pv, bins=256, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(pv.mean(), color='red', ls='--', label=f'Mean: {pv.mean():.3f}')
ax1.set_title('Pixel Intensity Histogram'); ax1.legend()
ax2.boxplot(pv, vert=False); ax2.set_title('Intensity Distribution')
plt.tight_layout(); plt.show()
print(f"Min: {pv.min():.4f}, Max: {pv.max():.4f}, Mean: {pv.mean():.4f}")

In [ ]:
lesion_sizes = {c: [] for c in range(1, config.NUM_CLASSES)}
for idx in range(min(len(dataset), 500)):
    _, mask, _ = dataset[idx]
    msk = mask.numpy() if torch.is_tensor(mask) else mask
    for c in range(1, config.NUM_CLASSES):
        sz = int((msk == c).sum())
        if sz > 0: lesion_sizes[c].append(sz)
fig, ax = plt.subplots(figsize=(10, 4))
bp = ax.boxplot([lesion_sizes[c] for c in range(1, config.NUM_CLASSES)],
                patch_artist=True, labels=[CLASS_NAMES[c] for c in range(1, config.NUM_CLASSES)])
for patch, color in zip(bp['boxes'], colors[1:]):
    patch.set_facecolor(color); patch.set_alpha(0.5)
ax.set_title('Lesion Size per Slice (pixels)'); ax.set_ylabel('Pixel Count'); ax.set_yscale('log')
plt.tight_layout(); plt.show()

## 3. Model Sanity Check

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = ConditionedSymFormer(in_channels=1, num_classes=3, T=1, input_size=(512, 512), bottleneck_type='mamba', use_kan=True).to(device)
x = torch.randn(2, 1, 512, 512).to(device)
meta = {'nihss': torch.tensor([10., 5.], device=device), 'age': torch.tensor([65., 70.], device=device),
        'sex': torch.tensor([0, 1], device=device), 'time': torch.tensor([3., 6.], device=device),
        'dsa': torch.tensor([0, 1], device=device)}
with torch.no_grad():
    out = model(x, metadata_dict=meta)
    print(f"Pred: {out['pred'].shape}, Multi-scale: {len(out.get('multiscale_preds', []))}, Params: {sum(p.numel() for p in model.parameters()):,}")
criterion = StrokeLoss(num_classes=3)
loss, d = criterion(out['pred'], torch.randint(0, 3, (2, 512, 512), device=device))
print(f"StrokeLoss: {loss.item():.4f}  " + '  '.join(f'{k}: {v:.4f}' for k,v in d.items()))
print("Sanity check passed \u2705")

## 4. Training

In [ ]:
DEVICES = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICES
config = load_config(DATASET)
config.create_directories()
config.print_config()

In [ ]:
dataset_class = get_dataset_class(config.DATASET_NAME)
root_path = config.DATA_PATHS.get(config.DATASET_NAME, config.BASE_PATH)
train_dataset = dataset_class(dataset_root=root_path, split='train', T=config.T, config=config)
val_dataset = dataset_class(dataset_root=root_path, split='val', T=config.T, config=config)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=config.PIN_MEMORY)
print(f"Train: {len(train_dataset)} samples / {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples / {len(val_loader)} batches")

In [ ]:
device = torch.device('cuda:0' if torch.cuda.device_count() > 0 else 'cpu')
multi_gpu = torch.cuda.device_count() > 1
model = ConditionedSymFormer(in_channels=config.NUM_CHANNELS, num_classes=config.NUM_CLASSES, T=config.T,
                             input_size=config.IMAGE_SIZE, bottleneck_type='mamba' if config.USE_MAMBA else 'symmetry', use_kan=config.USE_KAN)
if multi_gpu: model = torch.nn.DataParallel(model)
trainer = SymFormerTrainer(model=model, train_loader=train_loader, val_loader=val_loader, config=config, device=device, multi_gpu=multi_gpu)
print(f"Device: {device}  Multi-GPU: {multi_gpu}")

In [ ]:
NUM_EPOCHS = config.NUM_EPOCHS
# NUM_EPOCHS = 5
trainer.train(num_epochs=NUM_EPOCHS)

## 5. Evaluation

In [ ]:
evaluator = Evaluator(model=model, val_loader=val_loader, device=device, config=config, num_samples=5, output_dir='./evaluation_results')
summary = evaluator.run()
print(summary)

---
### Alternative: CLI Entry Point
```python
# %run ../scripts/train.py --devices "0" --dataset cpaisd
```